# Multimodal Input: Combining an Image with a Tabular Feature

Real problems rarely hand you only an image. A radiology model has the scan *and* the patient's age
and lab values; a product classifier has the photo *and* the seller's category and price. Those extra
fields are tabular, not visual, so they cannot simply be concatenated onto the pixels — the network
needs two inputs that meet somewhere inside.

This notebook builds that architecture. A convolutional branch processes the CIFAR-10 image, a second
input carries one scalar feature, the two are joined with a `Concatenate` layer, and a single softmax
reads the combined representation. Getting there requires stepping off the `Sequential` API and onto
Keras's **functional API**, which is the real subject of the notebook.

## Learning objectives

- Explain why a `Sequential` model cannot express a two-input architecture.
- Build a branching model with the Keras functional API, naming each `Input`.
- Merge a convolutional feature vector with tabular features using `Concatenate`.
- Train a multi-input model by passing a list of arrays to `model.fit`.
- Recognize target leakage in an auxiliary feature, and say what it does to a reported score.

## Background

You should be comfortable with the four-block CNN from `U2-2_CNN-4_Cifar.ipynb` — this notebook
reuses that exact stack as its image branch, so anything new here is in the wiring rather than the
convolutions.

The one genuinely new idea is the **functional API**. A `Sequential` model is a straight list of
layers, each consuming the previous one's output, which makes it structurally incapable of having two
inputs. In the functional API you instead call each layer on a tensor and capture the result:

```python
x = Conv2D(8, (3, 3))(img_input)     # call the layer on a tensor
x = BatchNormalization()(x)          # ...and again on the result
```

Because every intermediate tensor is a variable you hold, you can branch, merge, and reuse them
freely. The model is then defined by naming its endpoints: `Model(inputs=[...], outputs=...)`.

## This notebook covers

1. Loading CIFAR-10 and deriving an extra tabular feature
2. Building, training, and evaluating a two-input model
3. Review

**Prerequisites:** `U2-2_CNN-4_Cifar.ipynb` for the CNN architecture and the CIFAR-10 dataset.

**Dataset:** CIFAR-10, loaded via `tensorflow.keras.datasets.cifar10`, plus one derived binary
feature marking each image as a vehicle or an animal.

**References:** https://keras.io/guides/functional_api/

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import math

pd.set_option('display.max_columns',100)
pd.set_option('display.max_rows',100)

plt.style.use('dark_background')

import warnings
warnings.filterwarnings('ignore')

# Shared course helpers (msds565_helpers.py lives in the repo root).
# Notebooks sit two folders below the root, so '../..' points back to it.
import sys
sys.path.append('../..')
import msds565_helpers as helpers

## 1. Load and process the data

### 1.1 Load CIFAR-10

The same dataset as `U2-2_CNN-4_Cifar.ipynb`. Labels are flattened from `(N, 1)` to `(N,)` so the
scikit-learn metrics accept them directly.

In [ ]:
from tensorflow.keras.datasets import cifar10

(X_train, y_train), (X_test, y_test) = cifar10.load_data()

# Labels arrive as (N, 1); flatten to (N,) so sklearn's metrics accept them directly.
y_train = y_train.flatten()
y_test  = y_test.flatten()

# CIFAR-10's classes, in label order (0 = airplane, 1 = automobile, ...).
cifar_classes = ['airplane', 'automobile', 'bird', 'cat', 'deer',
                 'dog', 'frog', 'horse', 'ship', 'truck']

print("X_train.shape:", X_train.shape)
print("X_test.shape: ", X_test.shape)

### 1.2 Create the extra "type" feature

CIFAR-10 ships no tabular columns, so we manufacture one: a binary flag marking each image as a
**vehicle** (airplane, automobile, ship, truck) or an **animal** (the other six classes). It stands
in for the kind of side information a real dataset would carry — a sensor reading, a category code,
a patient's age.

> **Be honest about what this feature is.** It is derived directly from the label, which makes it
> **target leakage**: it tells the model something it could only know if it already knew the answer.
> A real deployment would have no way to compute it. That inflates the accuracy below — the flag
> immediately eliminates six of the ten candidate classes — so treat the score as a demonstration
> that the wiring works, *not* as an honest measure of image classification. The comparison against
> `U2-2_CNN-4_Cifar.ipynb`'s image-only result is not apples to apples.
>
> Leakage of exactly this shape is a common and expensive real-world mistake: a feature that is
> quietly a function of the target produces a model that looks excellent in validation and fails on
> deployment. Building it deliberately once is a good way to learn to recognize it.

In [ ]:
# Create extra type feature: 1 = vehicle, 0 = animal.
# Classes 0 (airplane), 1 (automobile), 8 (ship), and 9 (truck) are the vehicles.
vehicle_classes = [0, 1, 8, 9]

# Keras wants this input shaped (N, 1) to match Input(shape=(1,)).
type_train = np.isin(y_train, vehicle_classes).astype(int).reshape(-1, 1)
type_test  = np.isin(y_test,  vehicle_classes).astype(int).reshape(-1, 1)

print("type_train.shape:", type_train.shape)
print(f"vehicles in train: {type_train.sum()}   animals in train: {len(type_train) - type_train.sum()}")

## 2. Building a two-input model

### 2.1 Wiring the branches with the functional API

Three things to notice in the cell below.

**Two `Input` layers, each named.** `img_input` takes the 32×32×3 image; `type_input` takes a single
scalar. The `name=` arguments show up in `model.summary()` and make a branching graph far easier to
read.

**The image branch is the CIFAR CNN, unchanged.** The same four `Conv2D` → `BatchNormalization` →
`Activation` → `MaxPooling2D` → `Dropout` blocks from `U2-2_CNN-4_Cifar.ipynb`, written in functional
style. `GlobalAveragePooling2D` reduces the final 2×2×64 stack to a 64-element vector — one average
per feature map.

**`Concatenate` is where the modalities meet.** It joins that 64-element image vector with the
1-element type vector end to end, producing a 65-element vector that the final `Dense` softmax reads.

Where to merge is a genuine design decision. Concatenating *after* the convolutions, as here, lets
the image branch learn purely visual features and gives the tabular feature a say only at the final
decision. Merging earlier would let the extra feature modulate the convolutions themselves — more
expressive, harder to train, and harder to interpret.

In [ ]:
# -----------------------------------------------------------
# Build two-input CNN model
# -----------------------------------------------------------
from tensorflow.keras.models import Model
from tensorflow.keras.layers import *

dropout_rate = 0.2
n_classes  = np.unique(y_train).shape[0]

# Image input and CNN
img_input = Input(shape=X_train.shape[1:], name="image_input")

x = Conv2D(8, (3, 3), padding='same')(img_input)
x = BatchNormalization()(x)
x = Activation('relu')(x)
x = MaxPooling2D(pool_size=(2, 2))(x)
x = Dropout(dropout_rate)(x)

x = Conv2D(16, (3, 3), padding='same')(x)
x = BatchNormalization()(x)
x = Activation('relu')(x)
x = MaxPooling2D(pool_size=(2, 2))(x)
x = Dropout(dropout_rate)(x)

x = Conv2D(32, (3, 3), padding='same')(x)
x = BatchNormalization()(x)
x = Activation('relu')(x)
x = MaxPooling2D(pool_size=(2, 2))(x)
x = Dropout(dropout_rate)(x)

x = Conv2D(64, (3, 3), activation='relu', padding='same')(x)
x = MaxPooling2D(pool_size=(2, 2))(x)

x = GlobalAveragePooling2D()(x)

# Type input (scalar: 0 or 1)
type_input = Input(shape=(1,), name="type_input")

# Concatenate CNN features with type
concat = Concatenate()([x, type_input])

output = Dense(n_classes, activation='softmax')(concat)

# Final model
model = Model(inputs=[img_input, type_input], outputs=output)
model.summary()

### 2.2 Compile and configure training

Identical settings to `U2-2_CNN-4_Cifar.ipynb` — same optimizer, learning rate, batch size, and
early-stopping patience — so that any difference in the result comes from the added input rather than
from a changed training recipe.

In [ ]:
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping

epochs = 15
batch_size = 1000
initial_learning_rate = 0.01

optimizer = Adam(learning_rate=initial_learning_rate)

model.compile(
    optimizer=optimizer,
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

early_stopping = EarlyStopping(
    monitor='val_loss',
    patience=10,
    restore_best_weights=True
)

### 2.3 Train and evaluate

The only change to the training call is the input: a **list** `[X_train, type_train]` instead of a
single array, matching the order of the `inputs=[img_input, type_input]` list given to `Model`. Keras
routes each array to its corresponding `Input` layer. (Naming the inputs also allows a dictionary
here, which is safer when a model has many branches.)

`helpers.train_and_evaluate` handles a list-valued `X_train` without modification, since it passes
whatever it is given straight through to `model.fit` and `model.predict`.

In [ ]:
# Both inputs are passed as a LIST, in the same order as Model(inputs=[img_input, type_input]).
model, history = helpers.train_and_evaluate(
    model,
    [X_train, type_train], y_train,
    [X_test, type_test], y_test,
    epochs=epochs,
    batch_size=batch_size,
    callbacks=[early_stopping],
    class_names=cifar_classes
)

## 3. Review

| Piece | `Sequential` | Functional API |
|---|---|---|
| Layer wiring | Implicit — a list, each layer fed the previous | Explicit — `y = Layer(...)(x)` on tensors you hold |
| Inputs | Exactly one | Any number, each its own `Input` |
| Branching / merging | Not expressible | `Concatenate`, `Add`, `Multiply`, … |
| Model definition | The layer list itself | `Model(inputs=[...], outputs=...)` |
| `fit` call | One array | A list (or dict) of arrays, one per input |

**Takeaways**

- **The functional API is the whole unlock.** Once layers are called on tensors instead of stacked in
  a list, two inputs, skip connections, and shared branches all become expressible. Every non-trivial
  architecture in the rest of this unit — including the transfer-learning models — is built this way.
- **Modalities merge as vectors, not as pixels.** The image branch has to reduce to a flat feature
  vector before anything can be concatenated onto it. `GlobalAveragePooling2D` does that job here,
  turning 2×2×64 activations into 64 numbers that sit comfortably beside a tabular column.
- **Where you merge is a design decision.** Late fusion (after the convolutions, as here) keeps the
  branches independent and interpretable. Early fusion lets the side information shape the visual
  features themselves, at the cost of a harder optimization problem.
- **The reported accuracy here is inflated, and that is the real lesson.** The type flag is computed
  from the label, so it is target leakage — it rules out six classes for free, information no
  deployed model would have. A feature that is secretly a function of the target is one of the most
  common ways a project produces excellent validation numbers and a useless model. Before trusting
  any auxiliary feature, ask: *would this value actually be available, and correct, at prediction
  time?*
- **To see the honest version,** replace the derived flag with something genuinely independent of the
  label — image brightness, or mean saturation, both computable from the pixels alone — and watch
  how much less it helps. That gap between the leaked and non-leaked score is the size of the
  self-deception.

**Next:** `U2-2_CNN-6_TransferLearning.ipynb` stops training convolutional filters from scratch and
borrows them from a model trained on millions of images.